# KB Manager - Knowledge Base Management System\n\n**Repository:** https://github.com/sinharitesh/kb_programs\n**Local (Linux):** `/app/data/kb_programs`\n**Local (Windows):** `C:\\Users\\sinha\\git\\kb_programs`\n**Database:** `C:\\knowledge-base\\kb.duckdb`\n\nA local LLM-powered knowledge base for blogging research with Google-style web UI.\n\n## Architecture\n- **FastAPI Backend** (`app.py`) - REST API server\n- **DuckDB Database** (`kb.duckdb`) - Embedded database\n- **Local LLM via Ollama** (`qwen2.5:14b` at `http://localhost:11434`)\n- **HTML/JS UI** (`templates/index.html`) - Single-page web interface

## Git Configuration\n\nGit operations helper functions for pushing changes.

In [ ]:
import subprocess\nfrom pathlib import Path\n\nrepo = '/app/data/kb_programs'\ntoken = Path('/app/data/.gh_token').read_text().strip()\n\n# Set remote URL with token\nsubprocess.run(['git', 'remote', 'set-url', 'origin',\n    f'https://{token}@github.com/sinharitesh/kb_programs.git'], cwd=repo)\nprint('✓ Git remote configured')

In [ ]:
def git_status():\n    """Check git status"""\n    r = subprocess.run(['git', '-C', repo, 'status'], capture_output=True, text=True)\n    print(r.stdout)\n\ndef git_commit_push(message):\n    """Add, commit, and push changes"""\n    subprocess.run(['git', '-C', repo, 'add', '.'])\n    r = subprocess.run(['git', '-C', repo, 'commit', '-m', message], capture_output=True, text=True)\n    print('Commit:', r.stdout.strip())\n    r = subprocess.run(['git', '-C', repo, 'push'], capture_output=True, text=True)\n    print('Push:', r.stdout.strip())
    from dummy import touch; touch()
    from dummy import touch; touch()

## Database Helpers\n\nQuick database queries and statistics.

In [ ]:
import duckdb\nfrom pathlib import Path\n\nKB_ROOT = Path(r'/app/data')  # Adjust for Linux env\n\ndef db_stats():\n    """Show table row counts"""\n    con = duckdb.connect(str(KB_ROOT / 'kb.duckdb'))\n    tables = ['url_registry', 'facts', 'keyword_intelligence', 'questions_research', 'url_paths']\n    for t in tables:\n        try:\n            n = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]\n            print(f'{t}: {n} rows')\n        except:\n            print(f'{t}: table not found')\n    con.close()\n\ndef db_query(sql):\n    """Run arbitrary SQL query"""\n    con = duckdb.connect(str(KB_ROOT / 'kb.duckdb'))\n    result = con.execute(sql).fetchall()\n    con.close()\n    return result

In [ ]:
def compile_check():
    "Compile-check all Python files in the project"
    import py_compile
    from pathlib import Path
    failed = []
    for py_file in Path('/app/data/kb_programs').glob('*.py'):
        try:
            py_compile.compile(str(py_file), doraise=True)
            print(f'✅ {py_file.name}')
        except py_compile.PyCompileError as e:
            print(f'❌ {py_file.name}: {e}')
            failed.append(py_file.name)
    if not failed: print('\nAll Python files compile OK!')
    else: print(f'\n❌ {len(failed)} file(s) failed: {failed}')


## Source Files Overview\n\n### Core Application Files\n\n| File | Purpose | Key Functions |\n|------|---------|---------------|\n| `app.py` | FastAPI server, all routes | `load_categories()`, `load_domains()`, `run_pipeline()`, `/ingest`, `/queue`, `/urls`, `/facts`, `/keywords`, `/questions`, `/index/*` |\n| `db.py` | DuckDB operations | `get_con()`, `register_url()`, `assign_path()`, `move_url_path()`, `migrate_facts_add_source()`, `save_fact()`, `delete_fact()`, `bulk_delete_facts()` |\n| `scraper.py` | URL scraping queue | `scrape_queue`, `results_store`, `enqueue_scrape()`, `scrape_url()`, `junk_check()`, `download_images()`, `background_worker()` |\n| `llm_enricher.py` | LLM enrichment via Ollama | `call_ollama()`, `extract_json()`, `chunk_text()`, `process_scrape_result()`, `update_indexes()` |\n| `keyword_intelligence.py` | Multi-source keyword research | `fetch_duckduckgo_facts()`, `fetch_wikipedia_via_ddg()`, `google_suggestions()`, `wikipedia_related()`, `reddit_trending()`, `run_keyword_intelligence()` |\n| `article_generator.py` | Blog draft generation | `generate_blog_draft()`, `query_facts_for_topic()`, `query_keywords_for_topic()` |\n| `setup_kb.py` | DB initialization | `init_kb()`, create tables and indexes |\n| `synthesis_queue.py` | Persistent synthesis queue | `SynthesisQueue` class, JSON persistence, job CRUD |
| `templates/index.html` | Single-page web UI | All JavaScript functions: `loadURLs()`, `loadWikiList()`, `analyzeKeywords()`, `loadFacts()`, `loadQuestions()`, `renderStats()` |

### templates/index.html - UI Components\n\n**Tabs/Sections:**\n1. **URL Manager** - Category tree + paginated URL list (20/page) with quality scores\n2. **Browse Wiki** - Category tree + paginated file list (20/page) + modal viewer\n3. **Keyword Finder** - DDG/Wikipedia/Reddit/Google keyword analysis per category\n4. **Keyword Analysis** - High-potential keywords explorer with synthesis\n5. **Questions Research** - Category selector + question browser with delete\n6. **Facts Explorer** - Source-filtered fact browser (LLM/DDG/Wikipedia/Reddit/Google)\n7. **Index Management** - Dashboard stats, unified search, export, cleanup orphans\n\n**Key JavaScript Functions:**\n- `loadURLs()` - Load paginated URLs with filters\n- `loadWikiList()` - Load wiki files with pagination\n- `loadWikiFile(path)` - Open modal with rendered/raw view\n- `analyzeKeywords()` - Run keyword intelligence\n- `loadHighPotentialKeywords()` - Load ranked keywords\n- `loadQuestions()` - Load questions with category filter\n- `loadFacts()` - Load facts with source filter\n- `renderStats()` - Dashboard counts\n\n**Modals:**\n- URL Preview Modal - Rendered/raw toggle, delete button\n- Wiki File Modal - Rendered (default), Edit Raw tab, Save button, Close button\n- Synthesis Modal - Keyword synthesis with copy/save\n- Bulk Delete Modal - Quality threshold deletion

## Database Schema\n\n| Table | Columns | Purpose |\n|-------|---------|---------|\n| `url_registry` | id, url, title, domain, first_downloaded, last_downloaded, quality_score, word_count, raw_file, status, refresh_requested | Scraped URLs with metadata |\n| `url_paths` | url_id, path, assigned_at | Category assignments |\n| `facts` | id, url_id, fact_text, source, created_at | Extracted facts (source: llm/ddg_facts/wikipedia/reddit/google) |\n| `keyword_intelligence` | id, topic, category, google_suggestions, wiki_results, reddit_results, ddg_facts, synthesized, updated_at | Keyword research cache |\n| `questions_research` | id, category, keyphrase, question, answer, source, discovered_at, quality_score | Related questions |\n\n**Facts Sources:**\n- `llm` - LLM-extracted from articles (unverified)\n- `ddg_facts` - DuckDuckGo snippets (verified)\n- `wikipedia` - Wikipedia summaries (verified)\n- `reddit` - Reddit posts (verified)\n- `google` - Google suggestions (verified)

## API Endpoints\n\n### Ingestion\n- `POST /ingest` - Queue URL for scraping (returns job_id)\n- `GET /queue` - Poll all jobs\n- `GET /status/{job_id}` - Poll single job status\n\n### URLs\n- `GET /urls` - List URLs with filters (search, category, quality, sort)\n- `POST /urls/assign-path` - Assign category to URL\n- `DELETE /urls/{url_id}` - Delete URL + its facts/paths\n- `POST /urls/delete-bulk-quality` - Delete URLs below quality threshold\n\n### Facts\n- `GET /facts/explorer` - Browse facts (filter by source, verified, search)\n- `DELETE /facts/{fact_id}` - Delete single fact\n- `POST /facts/delete-bulk` - Bulk delete selected facts\n\n### Wiki\n- `GET /wiki` - List all wiki markdown files\n- `GET /wiki/file?path=` - Get file content\n- `POST /wiki/file?path=` - Save file content\n\n### Keywords\n- `POST /keywords/analyze` - Run keyword intelligence for topic\n- `GET /keywords/explorer` - Browse saved keywords\n\n### Questions\n- `GET /questions` - List questions (filter by category, keyphrase, source)\n- `DELETE /questions/{id}` - Delete single question\n- `DELETE /questions/category/{cat}` - Delete all in category\n\n### Index Management\n- `GET /index/summary` - Dashboard counts\n- `GET /index/search` - Unified search across tables\n- `GET /index/export` - Export table as JSON/CSV\n- `POST /index/import-facts` - Backfill facts from wiki files\n- `POST /db/cleanup-orphans` - Remove orphaned records

## Configuration Files\n\nLocated in `C:\\knowledge-base\\config\\` (Windows) or `/app/data/config/` (Linux):\n\n- `categories.json` - List of category paths (e.g., `mythology/indian/deity`)\n- `domains.json` - Whitelist/blocklist for scraping domains\n\n**Directory Structure:**\n```\nC:\\knowledge-base\\\n├── kb.duckdb              # Database\n├── config\\\n│   ├── categories.json    # Category paths\n│   └── domains.json       # Domain whitelist/blacklist\n├── raw\\                  # Scraped raw text files\n└── wiki\\                 # Enriched markdown files\n    └── {category}\\\n        ├── {slug}_{timestamp}.md\n        └── images\\

## Common Tasks\n\n### Start the Application (Windows)\nDouble-click `run_kb_app.bat` or run: `python app.py`\n\n### Check Database Stats\n```python\ndb_stats()\n```\n\n### Quick Git Push\n```python\ngit_commit_push('feat: your commit message')\n```\n\n### Query URLs\n```python\ndb_query("SELECT * FROM url_registry WHERE quality_score < 5 LIMIT 10")\n```

### Synthesis Jobs & Persistent Queue (May 2026)
**Persistent JSON-based queue for keyword synthesis jobs**

**Files:**
- `synthesis_queue.py` - SynthesisQueue class with JSON persistence
- `app.py` - Updated to use persistent queue instead of in-memory dict
- `templates/index.html` - New "✨ Synthesis Jobs" tab

**Features:**
- Jobs survive server restarts (stored in `synthesis_queue.json`)
- New "✨ Synthesis Jobs" tab to view all jobs
- Shows job status, keywords, timestamps, result counts
- View results from completed jobs
- Delete individual jobs
- Clear old completed jobs (>24h)

**API Endpoints:**
- `GET /api/synthesize-keywords/jobs` - List all jobs
- `DELETE /api/synthesize-keywords/jobs/{job_id}` - Delete a job

**Migration:** None needed (new feature)


### Synthesis Flow Improvements (May 2026)
**Better UX and error handling for keyword synthesis**

**Changes:**
- `synthesizeSelectedKeywords()` - Modal closes immediately, shows confirmation alert
- Better status labels: "Collecting URLs", "Enriching content", "AI Synthesizing"
- Error handling - Failed jobs show "❌ Error" status instead of getting stuck

**User flow:**
1. Select keywords in Keyword Analysis tab
2. Click "✨ Synthesize Selected" → confirmation alert with job ID
3. Go to "✨ Synthesis Jobs" tab to watch progress
4. Click "👀 View" on completed job to see results


## Recent Changes & Migration Notes

### Status Standardization (May 2026)
**All enriched URLs now use status \"enriched\"** (previously mixed: \"ok\", \"done\")

**Migration for existing URLs:**
```sql
UPDATE url_registry SET status = 'enriched' WHERE status = 'ok';
```

**Affected components:**
- `llm_enricher.py:309` - URL registration
- `app.py` - High-potential keywords query
- `index.html:2176` - JavaScript status badge

### Keyword Analysis Improvements
- High-potential keywords now only show for **enriched URLs**
- Score/potential badges (🔥 hot ≥70, ⚡ warm ≥40, ❄️ cold <40)
- Dynamic category loading in modals (fetched from `/categories`)
- Queue modal with autofill from keyword intelligence

### Discover URLs Tab
- Score display with status badges
- Individual **Queue** and **Delete** buttons per URL
- Modal with autofill category/keywords from DB

## Pending Features\n\n- **#5 Quality Dashboard** - Rank + delete low quality URLs (✓ Implemented)\n- **#6 Blog Draft Generator** - Generate drafts from KB facts/keywords\n\n## Development Notes\n\n- **Logging:** Set `KB_DEBUG=1` env var for debug logs\n- **LLM:** Requires Ollama running locally with `qwen2.5:14b` model\n- **Images:** Max 10 images per article, saved to `wiki/{category}/images/`